In [4]:
import pandas as pd 
from sklearn.manifold import TSNE 


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import imageio
import os


t = np.linspace(0, 2*np.pi, 200)
x_clean = np.cos(t)
y_clean = np.sin(t)

frames = []

for i in range(100):
    plt.figure()

    
    noise_scale = i / 50   

    x_noisy = x_clean + np.random.normal(0, noise_scale, size=x_clean.shape)
    y_noisy = y_clean + np.random.normal(0, noise_scale, size=y_clean.shape)

    plt.scatter(x_noisy, y_noisy, s=10)
    plt.scatter(x_clean , y_clean , c='red')
    plt.xlim(-3, 3)
    plt.ylim(-3, 3)
    plt.title(f"Noise level: {noise_scale:.2f}")

    filename = f"frame_{i}.png"
    plt.savefig(filename)
    plt.close()

    frames.append(filename)


with imageio.get_writer("perturbation.gif", mode= "I", duration=0.1) as writer:
    for filename in frames:
        writer.append_data(imageio.imread(filename))


for filename in frames:
    os.remove(filename)

/var/folders/51/kgb1jr9d4gx2d4flk26mbl180000gn/T/ipykernel_19867/2509371656.py:37: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(filename))


In [44]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import imageio
from io import BytesIO

title = "Demo2"

def sample_double_spiral(n=2000):
    n_half = n // 2
    t = np.linspace(0, 4*np.pi, n_half)

    x1 = t * np.cos(t)
    y1 = t * np.sin(t)

    x2 = -x1
    y2 = -y1

    x = np.concatenate([x1, x2])
    y = np.concatenate([y1, y2])

    x = x / (4*np.pi)
    y = y / (4*np.pi)

    return torch.tensor(np.stack([x, y], axis=1) , dtype=torch.float32)

def sample_s_curve(n=2000):
    t = np.random.uniform(-2, 2, n)
    x = t
    y = np.sin(2 * t)

    return torch.tensor(np.stack([x, y], axis=1),dtype=torch.float32)




def sample_data(n=1024):
    t = np.random.uniform(0, 2*np.pi, n)
    # x = np.stack([np.cos(t), np.sin(t)], axis=1)
    x = np.stack([np.cos(t)*np.sin(t), np.sin(t)], axis=1)
    return torch.tensor(x, dtype=torch.float32)



class ScoreNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),  # 2D point + sigma
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x, sigma):
        sigma = sigma.expand(x.shape[0], 1)
        inp = torch.cat([x, sigma], dim=1)
        return self.net(inp)


model = ScoreNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


sigmas = torch.tensor([0.5,0.4 ,0.35, 0.3,0.25 , 0.2, 0.1, 0.05, 0.01])



print("Training...")

for step in range(8000):
    x = sample_s_curve(512) #change this if u want to change the target manifold

    noise = torch.randn_like(x)
    sigma = sigmas[torch.randint(0, len(sigmas), (1,))]

    x_tilde = x + sigma * noise
    target = -noise / sigma

    pred = model(x_tilde, sigma)

    loss = ((pred - target)**2*sigma**2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")


print("Sampling...")

frames = []

x = torch.randn(1000, 2)  # start from noise
x_clean = sample_s_curve(512) # change this too, used to plot the clean curve on the gif frames
for sigma in sigmas:
    for _ in range(40):
        grad = model(x, sigma)

        step_size = 0.01 * (sigma**2)

        x = x + step_size * grad + torch.sqrt(torch.tensor(2 * step_size)) * torch.randn_like(x)


        fig, ax = plt.subplots(figsize=(5,5), dpi=100)

        ax.scatter(x[:,0].detach(), x[:,1].detach(), s=5, alpha=0.7)
        ax.scatter(x_clean[:,0].detach(), x_clean[:,1].detach(), s=5, alpha=0.7 , c='red')

        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect('equal')
        ax.set_title(f"sigma={sigma:.3f}")

        buf = BytesIO()
        plt.savefig(buf, format='png')
        plt.close(fig)

        buf.seek(0)
        frames.append(imageio.imread(buf))



frames = frames + frames[::-1]  

imageio.mimsave(title+'.gif', frames, duration=0.05 , loop=0)

print(f"Saved as {title}")

Training...
Step 0, Loss: 1.0154
Step 500, Loss: 0.6352
Step 1000, Loss: 0.9225
Step 1500, Loss: 0.9165
Step 2000, Loss: 0.9659
Step 2500, Loss: 0.4960
Step 3000, Loss: 0.9404
Step 3500, Loss: 0.4312
Step 4000, Loss: 0.8967
Step 4500, Loss: 0.4290
Step 5000, Loss: 0.9764
Step 5500, Loss: 0.4704
Step 6000, Loss: 0.5122
Step 6500, Loss: 0.4638
Step 7000, Loss: 0.4840
Step 7500, Loss: 0.9484
Sampling...


/var/folders/51/kgb1jr9d4gx2d4flk26mbl180000gn/T/ipykernel_19867/561888654.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = x + step_size * grad + torch.sqrt(torch.tensor(2 * step_size)) * torch.randn_like(x)
/var/folders/51/kgb1jr9d4gx2d4flk26mbl180000gn/T/ipykernel_19867/561888654.py:124: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frames.append(imageio.imread(buf))


Saved as Demo2
